# Optimizer Comparison — Fresh Server Run

**Designed to run on a completely fresh server (vast.ai or any GPU box with CUDA).**  
No pre-installed packages, no pre-cloned repos, no assumed directory structure.

**Only step required before running:** edit the `=== EDIT THIS ===` cell below.

Then: **Kernel → Restart & Run All**.

## ① Configuration — EDIT THIS CELL

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EDIT THIS CELL — everything else runs automatically
# ═══════════════════════════════════════════════════════════════════════════════

# GitHub access — needed to clone the private repo.
# Create a token at https://github.com/settings/tokens (scope: repo)
# Leave empty if the repo is public or you already cloned it.
GITHUB_TOKEN = ""  # e.g. "ghp_xxxxxxxxxxxx"

# Where to put everything (code, data, outputs, mlruns).
# Will be created if it doesn't exist. /root is fine for most GPU instances.
BASE_DIR = "/root/optimizer_run"

# Kaggle credentials — only needed for the Intel Image dataset.
# Paste the contents of your kaggle.json here as a dict, or leave None.
KAGGLE_CREDENTIALS = None
# KAGGLE_CREDENTIALS = {"username": "you", "key": "abc123"}

# ── Experiment settings ──────────────────────────────────────────────────────
NUM_SAMPLES = 40       # Optuna trials per optimizer
SEEDS       = [0, 1, 2, 3, 4]
NUM_WORKERS = 4        # DataLoader workers per trial
MOCK_RUN    = False    # True → smoke test: 4 trials, 2 epochs, tiny data subset

# Which tasks to run
RUN_REGRESSION             = True
RUN_TABULAR_CLASSIFICATION = True
RUN_IMAGE_CLASSIFICATION   = True

# Per-task dataset/model overrides — leave [] to use defaults:
#   regression:            datasets=[superconductivity, yearmsd]  models=[simple_mlp, attention_mlp]
#   tabular_classification: datasets=[adult, creditcard]          models=[simple_cls, attention_cls]
#   image_classification:  datasets=[places365, intel]            models=[resnet18, efficientnet_v2_s]
REGRESSION_DATASETS = []
REGRESSION_MODELS   = []
TABULAR_DATASETS    = []
TABULAR_MODELS      = []
IMAGE_DATASETS      = []
IMAGE_MODELS        = []
# ═══════════════════════════════════════════════════════════════════════════════

## ② Bootstrap — detect environment, derive paths

In [ ]:
import os, sys, subprocess, shutil, json
from pathlib import Path

BASE_DIR  = Path(BASE_DIR).expanduser().resolve()
CODE_DIR  = BASE_DIR / "code"
DATA_DIR  = BASE_DIR / "data"
OUT_DIR   = BASE_DIR / "outputs"
MFLOW_DIR = BASE_DIR / "mlruns"

for d in (BASE_DIR, CODE_DIR, DATA_DIR, OUT_DIR, MFLOW_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR  : {BASE_DIR}")
print(f"CODE_DIR  : {CODE_DIR}")
print(f"DATA_DIR  : {DATA_DIR}")
print(f"OUT_DIR   : {OUT_DIR}")
print(f"MFLOW_DIR : {MFLOW_DIR}")

# Write Kaggle credentials if provided
if KAGGLE_CREDENTIALS:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    kaggle_json_path = kaggle_dir / "kaggle.json"
    kaggle_json_path.write_text(json.dumps(KAGGLE_CREDENTIALS))
    kaggle_json_path.chmod(0o600)
    KAGGLE_JSON = str(kaggle_json_path)
    print(f"kaggle.json written to {kaggle_json_path}")
else:
    KAGGLE_JSON = None

print("Paths OK")

## ③ Install system packages & Python dependencies

In [ ]:
def _run(cmd, **kw):
    """Run a shell command, stream output, raise on error."""
    print(f"$ {' '.join(cmd) if isinstance(cmd, list) else cmd}")
    result = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if result.stdout.strip():
        print(result.stdout[-3000:])
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError(f"Command failed: {cmd}")
    return result

# System packages (apt)
apt_pkgs = ["git", "curl", "rsync"]
missing_apt = [p for p in apt_pkgs if not shutil.which(p)]
if missing_apt:
    _run(["apt-get", "install", "-y", "--no-install-recommends"] + missing_apt)
else:
    print("apt packages already present:", apt_pkgs)

print("System deps OK")

## ④ Clone the project repository

In [ ]:
REPO = "github.com/maximspbu/optimizers_comparison_tracking.git"
BRANCH = "sprint1"

if (CODE_DIR / "src").exists():
    print(f"Code already present at {CODE_DIR}, pulling latest ...")
    _run(["git", "-C", str(CODE_DIR), "pull", "--ff-only"])
else:
    if GITHUB_TOKEN:
        clone_url = f"https://{GITHUB_TOKEN}@{REPO}"
    else:
        clone_url = f"https://{REPO}"  # works for public repos
    print(f"Cloning branch '{BRANCH}' ...")
    _run(["git", "clone", "--depth=1", "--branch", BRANCH, clone_url, str(CODE_DIR)])

# Add code dir to path so `import src` works
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
os.chdir(str(CODE_DIR))

print(f"cwd: {os.getcwd()}")
print("Repo OK")

In [ ]:
# Install Python dependencies
req_file = CODE_DIR / "requirements.txt"
_run([sys.executable, "-m", "pip", "install", "--quiet",
      "--break-system-packages", "--ignore-installed",
      "-r", str(req_file)])
print("pip requirements OK")

In [ ]:
# Clone optimizer repos not on PyPI
EXTRA_REPOS = [
    ("https://github.com/nanowell/AdEMAMix-Optimizer-Pytorch", "AdEMAMix_Optimizer_Pytorch"),
    ("https://github.com/xinyuluo8561/Stacey",                "Stacey"),
]
for url, name in EXTRA_REPOS:
    dest = CODE_DIR / name
    if not dest.exists():
        print(f"Cloning {name} ...")
        _run(["git", "clone", "--depth=1", url, str(dest)])
    else:
        print(f"{name} already present")
    if str(dest) not in sys.path:
        sys.path.insert(0, str(dest))
print("Extra repos OK")

## ⑤ Environment verification

In [ ]:
import importlib

import torch
print(f"torch {torch.__version__}  CUDA: {torch.cuda.is_available()}")
GPU_NUM = torch.cuda.device_count()
for i in range(GPU_NUM):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory/1024**3:.1f} GB")
if GPU_NUM == 0:
    print("  No GPU detected — will run on CPU (slow!)")
    GPU_NUM = 0

for pkg in ("pytorch_lightning", "ray", "mlflow", "optuna", "openml"):
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg}: {getattr(m, '__version__', 'ok')}")
    except ImportError:
        print(f"  {pkg}: MISSING — re-run the pip cell above")

print("\nEnvironment check done.")

## ⑥ Configure logging & MLflow

In [ ]:
import logging

log_path = OUT_DIR / "run.log"
fmt = "%(asctime)s  %(levelname)-8s  %(name)s  %(message)s"
logging.basicConfig(
    level=logging.INFO,
    format=fmt,
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(str(log_path), mode="a"),
    ],
    force=True,
)
for noisy in ("ray", "ray.tune", "ray.air", "pytorch_lightning",
              "lightning", "absl", "urllib3"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

import mlflow
mlflow.set_tracking_uri(str(MFLOW_DIR))

from src.runner import ExperimentConfig, run_experiments, TASK_DEFAULTS

print(f"Logging → {log_path}")
print("Ready to run experiments.")

## ⑦ Run: Regression

In [ ]:
if RUN_REGRESSION:
    defaults = TASK_DEFAULTS["regression"]
    cfg_reg = ExperimentConfig(
        task_type   = "regression",
        datasets    = REGRESSION_DATASETS or defaults["datasets"],
        model_types = REGRESSION_MODELS   or defaults["model_types"],
        num_samples = NUM_SAMPLES,
        seeds       = SEEDS,
        batch_size  = defaults["batch_size"],
        num_epochs  = defaults["num_epochs"],
        gpu_num     = GPU_NUM,
        mock_run    = MOCK_RUN,
        output_dir  = str(OUT_DIR),
        mlflow_uri  = str(MFLOW_DIR),
        data_dir    = str(DATA_DIR),
        num_workers = NUM_WORKERS,
    )
    print(f"Regression  datasets={cfg_reg.datasets}  models={cfg_reg.model_types}")
    reg_results = run_experiments(cfg_reg)
    print("✓ Regression complete")
else:
    reg_results = {}
    print("Regression skipped")

## ⑧ Run: Tabular Classification

In [ ]:
if RUN_TABULAR_CLASSIFICATION:
    defaults = TASK_DEFAULTS["tabular_classification"]
    cfg_tab = ExperimentConfig(
        task_type   = "tabular_classification",
        datasets    = TABULAR_DATASETS or defaults["datasets"],
        model_types = TABULAR_MODELS   or defaults["model_types"],
        num_samples = NUM_SAMPLES,
        seeds       = SEEDS,
        batch_size  = defaults["batch_size"],
        num_epochs  = defaults["num_epochs"],
        gpu_num     = GPU_NUM,
        mock_run    = MOCK_RUN,
        output_dir  = str(OUT_DIR),
        mlflow_uri  = str(MFLOW_DIR),
        data_dir    = str(DATA_DIR),
        num_workers = NUM_WORKERS,
    )
    print(f"Tabular CLS  datasets={cfg_tab.datasets}  models={cfg_tab.model_types}")
    tab_results = run_experiments(cfg_tab)
    print("✓ Tabular classification complete")
else:
    tab_results = {}
    print("Tabular classification skipped")

## ⑨ Run: Image Classification

In [ ]:
if RUN_IMAGE_CLASSIFICATION:
    defaults = TASK_DEFAULTS["image_classification"]
    cfg_img = ExperimentConfig(
        task_type   = "image_classification",
        datasets    = IMAGE_DATASETS or defaults["datasets"],
        model_types = IMAGE_MODELS   or defaults["model_types"],
        num_samples = NUM_SAMPLES,
        seeds       = SEEDS,
        batch_size  = defaults["batch_size"],
        num_epochs  = defaults["num_epochs"],
        gpu_num     = GPU_NUM,
        mock_run    = MOCK_RUN,
        output_dir  = str(OUT_DIR),
        mlflow_uri  = str(MFLOW_DIR),
        data_dir    = str(DATA_DIR),
        num_workers = NUM_WORKERS,
        kaggle_json = KAGGLE_JSON,
    )
    print(f"Image CLS  datasets={cfg_img.datasets}  models={cfg_img.model_types}")
    img_results = run_experiments(cfg_img)
    print("✓ Image classification complete")
else:
    img_results = {}
    print("Image classification skipped")

## ⑩ Results

In [ ]:
import pandas as pd

results_dir = OUT_DIR / "results"

cross_csv = results_dir / "cross_summary.csv"
if cross_csv.exists():
    print("=== Cross-experiment summary ===")
    display(pd.read_csv(cross_csv, index_col=0))
else:
    print("cross_summary.csv not found yet")

In [ ]:
for csv_path in sorted(results_dir.glob("*_tuned_summary.csv")):
    exp_key = csv_path.stem.replace("_tuned_summary", "")
    print(f"\n=== {exp_key} — tuned ===")
    display(pd.read_csv(csv_path))

In [ ]:
import glob as _glob
from IPython.display import Image as IPImage, display as ipy_display

for subdir in ("best_run_plots", "tuned_run_plots", "default_run_plots", "varying_rs_run_plots"):
    pngs = sorted(_glob.glob(str(OUT_DIR / subdir / "*.png")))
    if not pngs:
        continue
    print(f"\n{'─'*60}  {subdir}  ({len(pngs)} plots)")
    for p in pngs:
        print(Path(p).name)
        ipy_display(IPImage(filename=p, width=900))

In [ ]:
from src.telemetry import load_telemetry, telemetry_summary

for jsonl in sorted((results_dir / "telemetry").glob("*.jsonl")):
    exp_key = jsonl.stem.replace("_telemetry", "")
    print(f"\n=== {exp_key} — telemetry ===")
    display(telemetry_summary(load_telemetry(str(jsonl))))

In [ ]:
# Start MLflow UI on port 5000
# Make sure port 5000 is open in your vast.ai instance network settings.
import subprocess as _sp
_sp.Popen(
    ["mlflow", "ui", "--backend-store-uri", str(MFLOW_DIR),
     "--host", "0.0.0.0", "--port", "5000"],
    stdout=_sp.DEVNULL, stderr=_sp.DEVNULL,
)
import socket
hostname = socket.gethostname()
print(f"MLflow UI → http://<instance-public-ip>:5000")
print(f"(hostname: {hostname})")